# Silver Layer - Instacart Dataset

Este notebook processa as tabelas Bronze e cria tabelas Silver com dados limpos, tipados e enriquecidos no schema `big_data.silver`.

## Transformações Aplicadas:
* **orders**: Casting de tipos, criação de features derivadas (is_first_order, period_of_day)
* **products_enriched**: Join com aisles, departments e prices para enriquecimento completo dos produtos
* **order_products**: União de prior + train datasets

## Qualidade de Dados:
* Filtros de NOT NULL em chaves primárias
* Validação de preços > 0
* Trim em strings
* Timestamp de processamento silver

In [0]:
bronze_schema = "big_data.bronze"
silver_schema = "big_data.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(f" Schema {silver_schema} pronto")

In [0]:
from pyspark.sql import functions as F

orders_silver = spark.table(f"{bronze_schema}.orders") \
    .withColumn("order_id",               F.col("order_id").cast("int")) \
    .withColumn("user_id",                F.col("user_id").cast("int")) \
    .withColumn("order_number",           F.col("order_number").cast("int")) \
    .withColumn("order_dow",              F.col("order_dow").cast("int")) \
    .withColumn("order_hour_of_day",      F.col("order_hour_of_day").cast("int")) \
    .withColumn("days_since_prior_order", F.col("days_since_prior_order").cast("double")) \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("user_id").isNotNull()) \
    .withColumn("is_first_order",
        F.when(F.col("order_number") == 1, True).otherwise(False)
    ) \
    .withColumn("period_of_day",
        F.when(F.col("order_hour_of_day").between(6, 11),  "morning")
         .when(F.col("order_hour_of_day").between(12, 17), "afternoon")
         .when(F.col("order_hour_of_day").between(18, 21), "evening")
         .otherwise("night")
    ) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.orders")

print(f"orders: {orders_silver.count():,} linhas")

In [0]:

products_silver = spark.table(f"{bronze_schema}.products") \
    .withColumn("aisle_id",      F.col("aisle_id").cast("int")) \
    .withColumn("department_id", F.col("department_id").cast("int")) \
    .join(
        spark.table(f"{bronze_schema}.aisles").select("aisle_id", "aisle"),
        on="aisle_id", how="left"
    ) \
    .join(
        spark.table(f"{bronze_schema}.departments").select("department_id", "department"),
        on="department_id", how="left"
    ) \
    .withColumn("product_name", F.trim(F.col("product_name"))) \
    .withColumn("product_name_normalized", 
        F.trim(F.regexp_replace(F.col("product_name"), "^'+|'+$", ""))
    ) \
    .join(
        spark.table(f"{bronze_schema}.prices").select(
            F.trim(F.regexp_replace(F.col("product_name"), "^'+|'+$", "")).alias("price_product_name"),
            F.expr("try_cast(price_usd as decimal(10,2))").alias("price_usd")
        ),
        on=F.col("product_name_normalized") == F.col("price_product_name"),
        how="left"
    ) \
    .drop("price_product_name", "product_name_normalized") \
    .withColumn("price_band",
        F.when(F.col("price_usd").isNull(), None)
         .when(F.col("price_usd") < 2.50, "Very Low")
         .when(F.col("price_usd") < 5.00, "Low")
         .when(F.col("price_usd") < 9.00, "Medium")
         .when(F.col("price_usd") < 15.00, "High")
         .when(F.col("price_usd") < 30.00, "Premium")
         .otherwise("Luxury")
    ) \
    .filter(F.col("product_id").isNotNull()) \
    .filter(F.col("product_name").isNotNull()) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.products_enriched")

In [0]:

prior = spark.table(f"{bronze_schema}.order_products_prior") \
    .withColumn("dataset", F.lit("prior"))

train = spark.table(f"{bronze_schema}.order_products_train") \
    .withColumn("dataset", F.lit("train"))

order_products_silver = prior.unionByName(train) \
    .withColumn("order_id",          F.col("order_id").cast("int")) \
    .withColumn("product_id",        F.col("product_id").cast("int")) \
    .withColumn("add_to_cart_order", F.col("add_to_cart_order").cast("int")) \
    .withColumn("reordered",         F.col("reordered").cast("boolean")) \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("product_id").isNotNull()) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

order_products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.order_products")

print(f"order_products: {order_products_silver.count():,} linhas")

In [0]:
print(f"Tabelas em {silver_schema}:\n")
spark.sql(f"SHOW TABLES IN {silver_schema}").show()

print("Contagem de linhas:")
for table in ["orders", "products_enriched", "order_products"]:
    count = spark.table(f"{silver_schema}.{table}").count()
    print(f"  {table}: {count:,} linhas")